In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .appName("Amazon Musical Instruments EDA")\
        .getOrCreate()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
0,application_1783355698266_0001,pyspark3,idle,Link,Link,✔


SparkSession available as 'spark'.


In [2]:
spark

In [3]:
df_review = spark.read.json(
    "s3://amazon-raw-data-group2/Musical_Instruments.jsonl"
)
df_review.printSchema()

root
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- attachment_type: string (nullable = true)
 |    |    |-- large_image_url: string (nullable = true)
 |    |    |-- medium_image_url: string (nullable = true)
 |    |    |-- small_image_url: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)

In [4]:
import pyspark.sql.functions as F
df_meta = spark.read.text("s3://amazon-raw-data-group2/meta_Musical_Instruments.jsonl")
df_meta = df_meta.select(
    F.get_json_object(F.col("value"), "$.main_category").alias("main_category"),
    F.get_json_object(F.col("value"), "$.title").alias("product_name"),
    F.get_json_object(F.col("value"), "$.average_rating").alias("average_rating"),
    F.get_json_object(F.col("value"), "$.rating_number").alias("rating_number"),
    F.get_json_object(F.col("value"), "$.price").alias("price"),
    F.get_json_object(F.col("value"), "$.store").alias("brand"),
    F.get_json_object(F.col("value"), "$.parent_asin").alias("parent_asin")
)
df_meta.printSchema()

root
 |-- main_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- average_rating: string (nullable = true)
 |-- rating_number: string (nullable = true)
 |-- price: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- parent_asin: string (nullable = true)

In [5]:
df_joined = df_review.join(
    df_meta,
    on="parent_asin",
    how="inner"
)

In [6]:
df = df_joined.select(
    "parent_asin",
    "asin",
    "helpful_vote",
    "rating",
    "text",
    "timestamp",
    "title",
    "user_id",
    "verified_purchase",
    "product_name",
    "price",
    "brand"
)

In [7]:
df.printSchema()

root
 |-- parent_asin: string (nullable = true)
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- product_name: string (nullable = true)
 |-- price: string (nullable = true)
 |-- brand: string (nullable = true)

In [8]:
print("Rows :", df.count())
print("Columns :", len(df.columns))

Rows : 3017439
Columns : 12

In [9]:
df.show(5, truncate=False)

+-----------+----------+------------+------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [10]:
#Cheacking Datatypes
df.dtypes

[('parent_asin', 'string'), ('asin', 'string'), ('helpful_vote', 'bigint'), ('rating', 'double'), ('text', 'string'), ('timestamp', 'bigint'), ('title', 'string'), ('user_id', 'string'), ('verified_purchase', 'boolean'), ('product_name', 'string'), ('price', 'string'), ('brand', 'string')]

In [11]:
#Missing Values
from pyspark.sql.functions import col, sum, when

missing = df.select([
    sum(when(col(c).isNull(),1).otherwise(0)).alias(c)
    for c in df.columns
])

missing.show(vertical=True)

-RECORD 0-------------------
 parent_asin       | 0      
 asin              | 0      
 helpful_vote      | 0      
 rating            | 0      
 text              | 0      
 timestamp         | 0      
 title             | 0      
 user_id           | 0      
 verified_purchase | 0      
 product_name      | 0      
 price             | 932112 
 brand             | 13495

In [12]:
#Check Empty String
from pyspark.sql.functions import trim

empty = df.select([
    sum(when(trim(col(c))=="",1).otherwise(0)).alias(c)
    for c in ["text","product_name","brand","price"]
])

empty.show()

+----+------------+-----+-----+
|text|product_name|brand|price|
+----+------------+-----+-----+
|2830|         216|    0|    0|
+----+------------+-----+-----+

In [13]:
from pyspark.sql.functions import col, when, regexp_replace

df = (
    df
    # Replace missing brand
    .withColumn(
        "brand",
        when(col("brand").isNull(), "Unknown Brand")
        .otherwise(col("brand"))
    )

    # Replace missing product name
    .withColumn(
        "product_name",
        when(col("product_name").isNull(), "Unknown Product")
        .otherwise(col("product_name"))
    )

    # Replace missing review text
    .withColumn(
        "text",
        when(col("text").isNull(), "No Review")
        .otherwise(col("text"))
    )

    # Clean price column
    .withColumn(
        "price",
        regexp_replace(col("price"), "[$,]", "")
    )

    # Convert price to double
    .withColumn(
        "price",
        col("price").cast("double")
    )

    # Replace missing price with 0
    .fillna({"price": 0.0})
)

In [14]:
from pyspark.sql.functions import sum, when, col

df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in ["brand", "product_name", "text", "price"]
]).show()

+-----+------------+----+-----+
|brand|product_name|text|price|
+-----+------------+----+-----+
|    0|           0|   0|    0|
+-----+------------+----+-----+

In [15]:
#unique Count
from pyspark.sql.functions import countDistinct

df.select(
    countDistinct("user_id"),
    countDistinct("asin"),
    countDistinct("brand"),
    countDistinct("parent_asin")
).show()

+-----------------------+--------------------+---------------------+---------------------------+
|count(DISTINCT user_id)|count(DISTINCT asin)|count(DISTINCT brand)|count(DISTINCT parent_asin)|
+-----------------------+--------------------+---------------------+---------------------------+
|                1762679|              259791|                26283|                     213571|
+-----------------------+--------------------+---------------------+---------------------------+

In [16]:
#Numerical Analysis
#Rating Statistics
df.describe(["rating"]).show()


+-------+------------------+
|summary|            rating|
+-------+------------------+
|  count|           3017439|
|   mean| 4.255534246094122|
| stddev|1.2781070691862728|
|    min|               1.0|
|    max|               5.0|
+-------+------------------+

In [17]:
df.select(
    "rating"
).summary().show()

+-------+------------------+
|summary|            rating|
+-------+------------------+
|  count|           3017439|
|   mean| 4.255534246094122|
| stddev|1.2781070691862728|
|    min|               1.0|
|    25%|               4.0|
|    50%|               5.0|
|    75%|               5.0|
|    max|               5.0|
+-------+------------------+

In [18]:
#Helpful Votes
df.describe(["helpful_vote"]).show()

+-------+------------------+
|summary|      helpful_vote|
+-------+------------------+
|  count|           3017439|
|   mean|1.1371795751297706|
| stddev|10.026837813346521|
|    min|                -1|
|    max|              4650|
+-------+------------------+

In [19]:
#Percentiles
df.approxQuantile(
    "helpful_vote",
    [0.25,0.5,0.75,0.95,0.99],
    0
)

[0.0, 0.0, 1.0, 4.0, 16.0]

In [20]:
#Rating Distribution
df.groupBy("rating")\
.count()\
.orderBy("rating")\
.show()

+------+-------+
|rating|  count|
+------+-------+
|   1.0| 264352|
|   2.0| 131441|
|   3.0| 197131|
|   4.0| 400387|
|   5.0|2024128|
+------+-------+

In [21]:
#Verified Purchase Analysis
df.groupBy("verified_purchase")\
.count()\
.show()

+-----------------+-------+
|verified_purchase|  count|
+-----------------+-------+
|             true|2780515|
|            false| 236924|
+-----------------+-------+

In [22]:
#Average RAting
from pyspark.sql.functions import avg

df.groupBy("verified_purchase")\
.agg(avg("rating").alias("avg_rating"))\
.show()

+-----------------+-----------------+
|verified_purchase|       avg_rating|
+-----------------+-----------------+
|             true|4.269449724241732|
|            false|4.092223666661039|
+-----------------+-----------------+

In [23]:
#Brand Analysis
df.groupBy("brand")\
.count()\
.orderBy("count",ascending=False)\
.show(20,False)

+---------------------+-----+
|brand                |count|
+---------------------+-----+
|Fender               |73518|
|Pyle                 |60712|
|Donner               |51324|
|YAMAHA               |45054|
|D'Addario            |40892|
|JIM DUNLOP           |38959|
|Behringer            |33321|
|OnStage              |32595|
|SNARK                |28146|
|D'Addario Accessories|26211|
|Ernie Ball           |24909|
|Neewer               |23109|
|ChromaCast           |20760|
|GLS Audio            |20492|
|Audio-Technica       |20011|
|Shure                |18635|
|Hola! Music          |17747|
|Mendini by Cecilio   |14703|
|Gator                |14145|
|Best Choice Products |14058|
+---------------------+-----+
only showing top 20 rows

In [24]:
#Average Rating
df.groupBy("brand")\
.agg(avg("rating").alias("avg_rating"))\
.orderBy("avg_rating",ascending=False)\
.show(20,False)

+---------------------------+----------+
|brand                      |avg_rating|
+---------------------------+----------+
|H&F Technologies           |5.0       |
|Happyfans                  |5.0       |
|D.S.                       |5.0       |
|Ashtonn                    |5.0       |
|Patricia Nash              |5.0       |
|Mingo Store                |5.0       |
|Highland Reeds             |5.0       |
|Love Dove                  |5.0       |
|Effects Pedals Flight Case |5.0       |
|MGGZXR                     |5.0       |
|Smartsails                 |5.0       |
|Kreminne                   |5.0       |
|Kansing                    |5.0       |
|poetry                     |5.0       |
|Buffalo Music Store/Dunlop |5.0       |
|Abbeyhorn                  |5.0       |
|Coolwin                    |5.0       |
|Rated:    PG    Format: DVD|5.0       |
|Laishalaiku                |5.0       |
|Color Me Mozart            |5.0       |
+---------------------------+----------+
only showing top

In [25]:
#Product Analysis
df.groupBy("product_name")\
.count()\
.orderBy("count",ascending=False)\
.show(20,False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|product_name                                                                                                                                                                                       |count|
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|GLS Audio Instrument Cable - Amp Cord for Bass & Electric Guitar - Straight to Right Angle 1/4 Inch Instrument Cable - Black/Grey Braided Tweed, 10ft                                              |9334 |
|BONAOK Wireless Bluetooth Karaoke Microphone, 3-in-1 Portable Handheld Mic Speaker Machine for All Smartphones, Gifts to Girls Boys Kids Adults All Age Q37(Black Gold)                

In [26]:
#Highest Rated products
df.groupBy("product_name")\
.agg(avg("rating").alias("avg_rating"))\
.orderBy("avg_rating",ascending=False)\
.show(20,False)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+
|product_name                                                                                                                                                                                        |avg_rating|
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+
|ESP LTD EC-401VF Tobacco Sunburst DiMarzio                                                                                                                                                          |5.0       |
|GEEKIA Hand Painted Glass Salt and Pepper Shaker Set With Holder Resin Figurine Sculpture in Decorative Bluebird,Cabin Kitchen Decor Table Decor Centerpieces &

In [27]:
#User Analysis
#Most Active Users
df.groupBy("user_id")\
.count()\
.orderBy("count",ascending=False)\
.show(20)

+--------------------+-----+
|             user_id|count|
+--------------------+-----+
|AG3S4FROO422V5KP7...|  497|
|AEYOYD3W6NQ6DMG3F...|  302|
|AF7CC34DK36SQJS7W...|  292|
|AH4GZFH3BWFUJTLJ7...|  257|
|AHGDGGMCSMMALHTX6...|  240|
|AG5ZVXXHEXDYUUODS...|  224|
|AGBG3KK74IKWJNQVM...|  215|
|AGQ7IEKV2MJP3TX4S...|  213|
|AFHPJTBQYQHQXIFBA...|  209|
|AFH6AUO7HH5PMJBRB...|  202|
|AFXVMXIAMROWJSIZ2...|  199|
|AFDAU5M5NRUN4LYLI...|  195|
|AEWQYNFYO55WCN2IV...|  192|
|AGSSF4YIHPCP2MIUI...|  190|
|AEVQNKERDSNYGGH7S...|  190|
|AG42JLA6HBMONDJEC...|  184|
|AFPBNLIVMV6PSXJFU...|  178|
|AGSVLMHAE6DKL34BW...|  177|
|AGID6DHXZOL34MVLP...|  175|
|AHEMAURZYRCARVTVF...|  169|
+--------------------+-----+
only showing top 20 rows

In [28]:
#Average Rating Per User
df.groupBy("user_id")\
.agg(avg("rating").alias("avg_rating"))\
.show()

+--------------------+------------------+
|             user_id|        avg_rating|
+--------------------+------------------+
|AEY3S4X43QALHDFWV...| 4.523809523809524|
|AFZSCJY32G7PIQIOB...|2.6666666666666665|
|AHY5RV5TCDAJ2T52C...|               5.0|
|AFFRUAJD5ZVLAP6TZ...|               4.0|
|AE2O4H7NUWKXHIDWO...|               1.0|
|AFXWICL6IA65YV6Q7...|               5.0|
|AE3EJOJAEZWCC4F67...| 4.166666666666667|
|AFBID75UGFINRNRTS...|               4.5|
|AFT36BY3XUFFBIBO7...|               5.0|
|AGCVQMJQ5YV57GITE...|               5.0|
|AEJBMWWKT7B2AWSAO...|               4.0|
|AFARBMS5FCXIGYLVJ...|               5.0|
|AG75T4FHLDTCMFK2P...| 3.857142857142857|
|AGT2VDLMW7SNNRIL2...|               4.0|
|AGOGAGKAIH4ND5SNV...|               5.0|
|AE64R32OG2A4WGC3K...|               5.0|
|AHUY7ET4WTNUVSW34...|3.8333333333333335|
|AEPIES2GGUGMWZGHB...|               5.0|
|AH5RUHPLEIYU4EZST...|               5.0|
|AFWRIUL6QQO4R37Y5...| 4.416666666666667|
+--------------------+------------

In [29]:
#Helpful Vote Analysis
df.orderBy("helpful_vote",ascending=False)\
.select(
    "rating",
    "helpful_vote",
    "text"
)\
.show(20,truncate=False)

+------+------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [30]:
#Average Helpful vote Rating
df.groupBy("rating")\
.agg(avg("helpful_vote"))\
.show()

+------+------------------+
|rating| avg(helpful_vote)|
+------+------------------+
|   1.0|1.4744431666868418|
|   4.0|1.3571619458174216|
|   3.0| 1.268410346419386|
|   2.0|1.2636315913603822|
|   5.0|1.0286266481171151|
+------+------------------+

In [31]:
#Review Length Analysis
from pyspark.sql.functions import length

df = df.withColumn(
    "review_length",
    length("text")
)

In [32]:
#Statistics
df.describe(["review_length"]).show()

+-------+------------------+
|summary|     review_length|
+-------+------------------+
|  count|           3017439|
|   mean|238.59247394893484|
| stddev|395.96620370662265|
|    min|                 0|
|    max|             32962|
+-------+------------------+

In [33]:
#Average review length by rating
df.groupBy("rating")\
.agg(avg("review_length"))\
.show()

+------+------------------+
|rating|avg(review_length)|
+------+------------------+
|   1.0|251.75775481176612|
|   4.0|320.77232777287975|
|   3.0| 326.1679086495782|
|   2.0| 327.3194132728753|
|   5.0|206.32662707101528|
+------+------------------+

In [34]:
#Longest Reviw
df.orderBy("review_length",ascending=False)\
.select(
    "rating",
    "review_length",
    "text"
)\
.show(20,False)

+------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [35]:
#Price Analysis
from pyspark.sql.functions import regexp_replace

df = df.withColumn(
    "price",
    regexp_replace("price","\\$","")
)

df = df.withColumn(
    "price",
    col("price").cast("double")
)

In [36]:
#Statistics
df.describe(["price"]).show()

+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|           3017439|
|   mean|53.033821141036796|
| stddev|132.41093755330195|
|    min|               0.0|
|    max|          15499.95|
+-------+------------------+

In [37]:
#Average price by brand

df.groupBy("brand")\
.agg(avg("price").alias("avg_price"))\
.show()

+--------------------+------------------+
|               brand|         avg_price|
+--------------------+------------------+
|         carrotmusic|               0.0|
|Adkins Profession...|10.452269662921344|
|       Reunion Blues|119.91758883248741|
|            JBL Bags| 60.79871892925428|
|          Danelectro| 19.00082530120479|
|               XANAD| 20.81068376068377|
|Homebrew Electronics|               0.0|
|          Pioneer DJ| 178.8609597352454|
|           WingPower|  7.95127272727273|
|              DIADEM|               0.0|
|              LEKATO| 39.30969745222917|
|            LanSenSu|13.573333333333336|
|            PRO MARK|  7.31340579710145|
|           Strumhard| 27.92172413793104|
|               T-Rex|111.26323353293412|
|               9HORN|               0.0|
|            LavoHome|               0.0|
|            Nyvoetaa|             49.99|
|        Sony Optiarc|               0.0|
|Saxophone Accesso...|               0.0|
+--------------------+------------

In [38]:
#Price vs Rating

df.groupBy("rating")\
.agg(avg("price").alias("avg_price"))\
.show()

+------+------------------+
|rating|         avg_price|
+------+------------------+
|   1.0| 49.55571416142116|
|   4.0| 51.36775000187353|
|   3.0| 49.85330374218155|
|   2.0|50.233317305863615|
|   5.0|54.309233250068445|
+------+------------------+

In [39]:
#Time Analysis

#Convert timestamp

from pyspark.sql.functions import from_unixtime

df = df.withColumn(
    "date",
    from_unixtime(col("timestamp")/1000)
)

In [41]:
#Extract

from pyspark.sql.functions import year,month

df = df.withColumn("year",year("date"))

df = df.withColumn("month",month("date"))

In [42]:
#Reviews over years

df.groupBy("year")\
.count()\
.orderBy("year")\
.show()

+----+------+
|year| count|
+----+------+
|1999|     2|
|2000|    29|
|2001|    46|
|2002|    70|
|2003|   186|
|2004|   425|
|2005|   783|
|2006|  1211|
|2007|  3035|
|2008|  4379|
|2009|  7605|
|2010| 17009|
|2011| 32394|
|2012| 51822|
|2013|116884|
|2014|174144|
|2015|241910|
|2016|271023|
|2017|261260|
|2018|273353|
+----+------+
only showing top 20 rows

In [43]:
#Average rating by year

df.groupBy("year")\
.agg(avg("rating"))\
.orderBy("year")\
.show()

+----+------------------+
|year|       avg(rating)|
+----+------------------+
|1999|               3.0|
|2000| 4.172413793103448|
|2001| 3.717391304347826|
|2002|4.3428571428571425|
|2003|3.2311827956989245|
|2004| 3.891764705882353|
|2005| 3.974457215836526|
|2006|3.9826589595375723|
|2007| 4.120263591433279|
|2008| 4.143868463119434|
|2009|4.1922419460880995|
|2010| 4.187606561232289|
|2011| 4.199265296042477|
|2012| 4.237563197097757|
|2013| 4.295848875808494|
|2014| 4.316588570378538|
|2015| 4.324628994254061|
|2016| 4.326699210030145|
|2017|  4.28540534333614|
|2018| 4.271659722044389|
+----+------------------+
only showing top 20 rows

In [44]:
#Monthly trend

df.groupBy("year","month")\
.count()\
.orderBy("year","month")\
.show()

+----+-----+-----+
|year|month|count|
+----+-----+-----+
|1999|    8|    1|
|1999|   10|    1|
|2000|    1|    4|
|2000|    2|    1|
|2000|    4|    1|
|2000|    5|    1|
|2000|    6|    1|
|2000|    7|    5|
|2000|    8|    2|
|2000|    9|    2|
|2000|   10|    3|
|2000|   11|    4|
|2000|   12|    5|
|2001|    1|    2|
|2001|    2|    2|
|2001|    3|    5|
|2001|    4|    4|
|2001|    5|    1|
|2001|    7|    4|
|2001|    8|    6|
+----+-----+-----+
only showing top 20 rows

In [45]:
#Text Analysis

#Review word count

from pyspark.sql.functions import split,size

df = df.withColumn(
    "word_count",
    size(split("text"," "))
)

In [46]:
#Statistics

df.describe(["word_count"]).show()

+-------+------------------+
|summary|        word_count|
+-------+------------------+
|  count|           3017439|
|   mean|44.759454955013176|
| stddev| 73.02163977725263|
|    min|                 1|
|    max|              6284|
+-------+------------------+

In [47]:
#Average word count

df.groupBy("rating")\
.agg(avg("word_count"))\
.show()

+------+------------------+
|rating|   avg(word_count)|
+------+------------------+
|   1.0|47.312885849170804|
|   4.0| 60.43380779096224|
|   3.0| 61.66103251137569|
|   2.0|61.605838360937604|
|   5.0| 38.58546692699276|
+------+------------------+

In [48]:
#Correlation
# Between
# -Rating
# -Helpful votes
# -Price
# -Review length

df.stat.corr("rating","helpful_vote")


-0.014127523130376208

In [49]:
df.stat.corr("rating","price")


0.013126441179162274

In [50]:
df.stat.corr("rating","review_length")


-0.07618398662124679

In [51]:
df.stat.corr("helpful_vote","review_length")

0.19240721565245958

In [52]:
#Outlier Detection
df.approxQuantile(
    "helpful_vote",
    [0.25,0.75],
    0
)

[0.0, 1.0]

In [53]:
#Top Insights
#Top 10 brands by reviews

df.groupBy("brand")\
.count()\
.orderBy(col("count").desc())\
.show(10,False)

+---------------------+-----+
|brand                |count|
+---------------------+-----+
|Fender               |73518|
|Pyle                 |60712|
|Donner               |51324|
|YAMAHA               |45054|
|D'Addario            |40892|
|JIM DUNLOP           |38959|
|Behringer            |33321|
|OnStage              |32595|
|SNARK                |28146|
|D'Addario Accessories|26211|
+---------------------+-----+
only showing top 10 rows

In [54]:
#Top products

df.groupBy("product_name")\
.count()\
.orderBy(col("count").desc())\
.show(10,False)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|product_name                                                                                                                                                                                  |count|
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|GLS Audio Instrument Cable - Amp Cord for Bass & Electric Guitar - Straight to Right Angle 1/4 Inch Instrument Cable - Black/Grey Braided Tweed, 10ft                                         |9334 |
|BONAOK Wireless Bluetooth Karaoke Microphone, 3-in-1 Portable Handheld Mic Speaker Machine for All Smartphones, Gifts to Girls Boys Kids Adults All Age Q37(Black Gold)                       |7380 |
|Blue

In [55]:
#Top expensive products

df.orderBy(col("price").desc())\
.select("product_name","brand","price")\
.show(10,False)

+------------------------------------------------+------+--------+
|product_name                                    |brand |price   |
+------------------------------------------------+------+--------+
|Sony C-800G Large-Diaphragm Condenser Microphone|Sony  |15499.95|
|Sony C-800G Large-Diaphragm Condenser Microphone|Sony  |15499.95|
|Sony C-800G Large-Diaphragm Condenser Microphone|Sony  |15499.95|
|Sony C-800G Large-Diaphragm Condenser Microphone|Sony  |15499.95|
|Sony C-800G Large-Diaphragm Condenser Microphone|Sony  |15499.95|
|Sony C-800G Large-Diaphragm Condenser Microphone|Sony  |15499.95|
|Sony C-800G Large-Diaphragm Condenser Microphone|Sony  |15499.95|
|Martin D-45 Natural                             |MARTIN|9699.0  |
|Martin D-45 Natural                             |MARTIN|9699.0  |
|Conn Alto Horn, Nickel,Silver (8DS)             |Conn  |9235.75 |
+------------------------------------------------+------+--------+
only showing top 10 rows

In [56]:
#Highest rated brands (minimum 100 reviews)

from pyspark.sql.functions import count

brand_rating = df.groupBy("brand").agg(
    avg("rating").alias("avg_rating"),
    count("*").alias("reviews")
)

brand_rating.filter(col("reviews") >= 100)\
.orderBy(col("avg_rating").desc())\
.show(20, False)

+--------------------------------------------------------------------------------+------------------+-------+
|brand                                                                           |avg_rating        |reviews|
+--------------------------------------------------------------------------------+------------------+-------+
|Disc-O-Files                                                                    |4.980582524271845 |103    |
|TOOBOSS                                                                         |4.9393939393939394|132    |
|BIRCH & SMITH                                                                   |4.919117647058823 |136    |
|Coleman Custom Picks                                                            |4.913385826771654 |127    |
|Zither USA                                                                      |4.907258064516129 |248    |
|Healing Lama                                                                    |4.90566037735849  |106    |
|WARNER SP

In [61]:
#Create a Sentiment Column
from pyspark.sql.functions import when, col

df = df.withColumn(
    "sentiment",
    when(col("rating") >= 4, "Positive")
    .when(col("rating") == 3, "Neutral")
    .otherwise("Negative")
)

df.select("rating", "sentiment").show(10)

+------+---------+
|rating|sentiment|
+------+---------+
|   5.0| Positive|
|   4.0| Positive|
|   5.0| Positive|
|   5.0| Positive|
|   5.0| Positive|
|   5.0| Positive|
|   5.0| Positive|
|   4.0| Positive|
|   5.0| Positive|
|   5.0| Positive|
+------+---------+
only showing top 10 rows

In [62]:
#Count Positive, Neutral, and Negative Reviews
sentiment_df = (
    df.groupBy("sentiment")
      .count()
      .orderBy("sentiment")
)

sentiment_df.show()

+---------+-------+
|sentiment|  count|
+---------+-------+
| Negative| 395793|
|  Neutral| 197131|
| Positive|2424515|
+---------+-------+

In [63]:
#Percentage of Each Sentiment
from pyspark.sql.functions import count, round

total = df.count()

df.groupBy("sentiment") \
  .agg(
      count("*").alias("Reviews"),
      round((count("*") / total) * 100, 2).alias("Percentage")
  ) \
  .show()

+---------+-------+----------+
|sentiment|Reviews|Percentage|
+---------+-------+----------+
| Positive|2424515|     80.35|
|  Neutral| 197131|      6.53|
| Negative| 395793|     13.12|
+---------+-------+----------+

In [65]:
output_path = "s3://musical-instrumental-data/updated-data/amazon-musical-instruments/processed_reviews/"

df.write \
    .mode("overwrite") \
    .parquet(output_path)